# 03 · Fusion Models

Notebooks 01 and 02 produced three feature sets per patient:

| Modality | Shape | Source |
|---|---|---|
| WSI | `(N patches, 1536)` | H-optimus-0, one bag per slide |
| MRI | `(2048,)` | MRI-PTPCa |
| Clinical | `(22,)` | one-hot / numeric encoding |

**A note on multi-slide patients:** CHIMERA patients have 1-12 WSI slides each, not always one.
The fusion architectures below don't need to know that -- they just take one WSI bag per *row*
of the batch. Notebook 04 is where that distinction actually gets handled: each of a patient's
slides becomes its own training row (with that patient's label copied onto every one of their
slides, and their MRI/clinical vectors repeated), and predictions only get combined back to the
patient level at evaluation time -- mean probability (+ majority vote for the hard label) for
classification, mean risk score for survival, following the same convention as
[nnMIL](https://arxiv.org/abs/2511.14907).

...and two label sets sharing the same `BCR` / `time_to_follow-up/BCR` fields: binary
**classification** (did BCR occur?) and censoring-aware **survival analysis** (time-to-event).

This notebook builds and explains three ways to combine the three modalities:

- **Early fusion** -- concatenate modality vectors, then one shared network does everything.
- **Intermediate fusion** -- each modality is projected into a shared space first, then a
  single self-attention pass lets them interact, *then* a shared network finishes the job.
- **Late fusion** -- each modality gets its own fully independent model all the way to a
  prediction; only the final predictions are combined.

**What this notebook does:** define and explain each architecture, sanity-check all 3
strategies x 2 tasks on dummy tensors (correct output shapes, gradients actually flow), and
compare parameter counts. **What it doesn't do:** load the real dataset or actually train --
that's notebook 04.

**Where the code ends up:** the three architectures are the lesson, so they're fully defined
and explained here. But notebook 04 needs the *exact same* classes to train them for real, so
the identical, finalized versions of everything below also live in `src/fusion_models.py`,
which notebook 04 imports -- avoiding notebook 04 having to redefine ~150 lines of model code
from scratch.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> repo root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)

## Shared building block 1 · MIL pooling (ABMIL)

WSI arrives as a *bag* of N patch embeddings, where N differs per slide -- no fusion strategy
below can use that directly, it first needs to become a single vector. The simplest option is
mean-pooling every patch equally. **ABMIL** does better: it learns an attention score per
patch (e.g. so tumor-region patches can end up weighted more than background patches) and
takes a weighted sum instead of a plain average.

**Important:** all three fusion strategies below use this *exact same* `ABMIL` definition. That
way, when we compare early vs. intermediate vs. late, the only thing that differs is the fusion
mechanism itself -- not incidentally-different WSI pooling quality.

In [ ]:
class ABMIL(nn.Module):
    """Attention-based MIL pooling: turns a bag of N patch embeddings into one vector,
    weighting each patch by a learned attention score instead of averaging them uniformly."""
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (N, D) patch embeddings for one WSI -> (D,) bag-level embedding."""
        attn_scores = self.attention(x)              # (N, 1)
        attn_weights = F.softmax(attn_scores, dim=0)  # (N, 1)
        return (attn_weights * x).sum(dim=0)          # (D,)

In [ ]:
# Sanity check: a dummy bag of 50 patches, 1536-dim each (matching H-optimus-0's output).
dummy_bag = torch.randn(50, 1536)
abmil = ABMIL(input_dim=1536)
pooled = abmil(dummy_bag)
print("pooled shape:", pooled.shape)

attn_weights = F.softmax(abmil.attention(dummy_bag), dim=0)
print("attention weights sum to:", attn_weights.sum().item())  # should be 1.0

## Shared building block 2 · task heads (classification vs. survival)

Both tasks read the same fused embedding, but need different output shapes:

- **Classification**: one logit -> `BCEWithLogitsLoss` -> AUROC.
- **Survival**: one hazard logit *per time bin* -> discrete-time NLL loss -> C-index.

To keep comparisons fair -- across fusion strategies, and across tasks -- the head
architecture itself should be identical everywhere; only the final output dimension should
change. `PredictionHead` matches HIMF-Surv's own `MLPPredictionHead` exactly: a 3-layer MLP
(`input -> 64 -> 32 -> output`), with BatchNorm + ReLU after the first two layers.

In [ ]:
class PredictionHead(nn.Module):
    """Shared 3-layer MLP head for *both* tasks -- only `output_dim` differs."""
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def build_task_head(input_dim: int, task: str, num_time_bins: int = 15) -> nn.Module:
    """Classification -> one logit. Survival -> one hazard logit per time bin. Same
    `PredictionHead` architecture either way -- only `output_dim` changes."""
    if task == "classification":
        return PredictionHead(input_dim, output_dim=1)
    elif task == "survival":
        return PredictionHead(input_dim, output_dim=num_time_bins)
    raise ValueError(f"Unknown task: {task!r} (expected 'classification' or 'survival')")

In [ ]:
# Sanity check: same input embedding, two different heads.
dummy_embedding = torch.randn(4, 128)  # (batch=4, fused_dim=128)

cls_head = build_task_head(128, task="classification")
surv_head = build_task_head(128, task="survival", num_time_bins=15)

print("classification head output:", cls_head(dummy_embedding).shape)  # (4, 1)
print("survival head output:", surv_head(dummy_embedding).shape)       # (4, 15)

## Survival-specific utilities

Ported from HIMF-Surv's `model.py`, unchanged:

- **`discretize_time`**: buckets continuous follow-up months into `num_time_bins` equal-width
  bins, so the model predicts a hazard *per bin* instead of a single continuous time.
- **`NLLLoss`**: turns per-bin hazard logits into a survival curve
  (`survival = cumprod(1 - hazard)`), then scores how likely the *observed* outcome was --
  the hazard-at-event term if BCR occurred in that bin, or the still-surviving term if the
  patient was censored (no event observed yet at last follow-up).
- **`concordance_index`**: the fraction of comparable patient pairs (one had an earlier
  observed event than the other) the model ranks correctly by predicted risk. 0.5 = random,
  1.0 = perfect -- this is the metric notebook 04 will report for the survival task, the way
  AUROC will be reported for classification.

In [ ]:
def discretize_time(time: torch.Tensor, num_bins: int, max_time: float, device: str) -> torch.Tensor:
    """Bucket continuous follow-up time into `num_bins` equal-width bins."""
    time = torch.as_tensor(time, dtype=torch.float32, device=device)
    bins = torch.linspace(0, max_time, num_bins + 1, device=device)
    discretized = torch.bucketize(time, bins, right=True) - 1
    return torch.clamp(discretized, 0, num_bins - 1)


class NLLLoss(nn.Module):
    """Discrete-time negative log-likelihood survival loss."""
    def __init__(self, reduction: str = "mean"):
        super().__init__()
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, y_time: torch.Tensor, y_event: torch.Tensor) -> torch.Tensor:
        y_time = y_time.long().unsqueeze(1)
        y_event = y_event.long().unsqueeze(1)
        num_bins = logits.shape[1]
        y_time = torch.clamp(y_time, 0, num_bins - 1)

        hazards = torch.sigmoid(logits)
        survival = torch.cumprod(1 - hazards, dim=1)
        survival_padded = torch.cat([torch.ones_like(y_event, dtype=torch.float), survival], dim=1)

        s_prev = torch.gather(survival_padded, 1, y_time).clamp(min=1e-7)
        h_this = torch.gather(hazards, 1, y_time).clamp(min=1e-7)
        log_lik_event = torch.log(s_prev) + torch.log(h_this)

        y_time_next = torch.clamp(y_time + 1, 0, num_bins)
        log_lik_censored = torch.log(torch.gather(survival_padded, 1, y_time_next).clamp(min=1e-7))

        neg_log_lik = -(y_event * log_lik_event + (1 - y_event) * log_lik_censored)
        return neg_log_lik.mean() if self.reduction == "mean" else neg_log_lik.sum()


def concordance_index(event_times, predicted_risk, event_observed) -> float:
    """Fraction of comparable pairs the model ranks correctly by predicted risk."""
    if isinstance(event_times, torch.Tensor):
        event_times = event_times.detach().cpu().numpy()
    if isinstance(predicted_risk, torch.Tensor):
        predicted_risk = predicted_risk.detach().cpu().numpy()
    if isinstance(event_observed, torch.Tensor):
        event_observed = event_observed.detach().cpu().numpy()

    n = len(event_times)
    concordant, permissible = 0.0, 0
    for i in range(n):
        if event_observed[i] == 0:
            continue
        for j in range(n):
            if i == j or event_times[i] >= event_times[j]:
                continue
            permissible += 1
            if predicted_risk[i] > predicted_risk[j]:
                concordant += 1
            elif predicted_risk[i] == predicted_risk[j]:
                concordant += 0.5
    return concordant / permissible if permissible > 0 else 0.5

In [ ]:
# Sanity check on synthetic hazard logits / times / events.
dummy_logits = torch.randn(6, 15, requires_grad=True)
dummy_time = torch.rand(6) * 60
dummy_event = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0, 0.0])

dummy_bins = discretize_time(dummy_time, num_bins=15, max_time=60.0, device="cpu")
loss = NLLLoss()(dummy_logits, dummy_bins, dummy_event)
loss.backward()
print("NLL loss:", loss.item(), "| gradient flows:", dummy_logits.grad is not None)

ci = concordance_index(dummy_time, torch.randn(6), dummy_event)
print("concordance_index (random risk, sanity range check):", ci)
assert 0.0 <= ci <= 1.0

## Early fusion

Concatenate modality vectors *before* any joint modeling, then let a single shared trunk do
all the work -- no per-modality projection layers (that would start to look like
intermediate fusion's per-modality encoders).

Two parameter-free, non-modality-specific normalization steps keep this practical without
compromising what "early" means:

1. **L2-normalize each modality vector before concatenation.** wsi (1536-dim) and mri
   (2048-dim) would otherwise dwarf clinical (22-dim) by sheer feature count -- a common fix in
   naive concatenation-based fusion baselines. This doesn't reduce dimensionality or add a
   learned per-modality transform, it just puts every modality's vector on the same overall
   scale before they're concatenated.
2. **BatchNorm the concatenated vector.** This is a *different* problem from (1): it normalizes
   each of the ~3600 individual concatenated features' distribution across the batch (same
   purpose as the BatchNorm inside `PredictionHead`, just applied to the raw multimodal input
   instead of a hidden layer -- the `trunk` below has no BatchNorm of its own, so without this,
   the first `Linear` would see completely unnormalized, wildly different-scale features).

In [ ]:
class EarlyFusionModel(nn.Module):
    def __init__(self, wsi_dim=1536, mri_dim=2048, clinical_dim=22, hidden_dim=256,
                 task="classification", num_time_bins=15):
        super().__init__()
        self.abmil = ABMIL(wsi_dim, hidden_dim=128)

        fused_dim = wsi_dim + mri_dim + clinical_dim
        self.input_norm = nn.BatchNorm1d(fused_dim)
        self.trunk = nn.Sequential(
            nn.Linear(fused_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
        )
        self.head = build_task_head(hidden_dim // 2, task, num_time_bins)

    def forward(self, batch: dict) -> torch.Tensor:
        wsi_pooled = torch.stack([self.abmil(w) for w in batch["wsi"]])  # (B, wsi_dim)

        wsi_pooled = F.normalize(wsi_pooled, p=2, dim=1)
        mri = F.normalize(batch["mri"], p=2, dim=1)
        clinical = F.normalize(batch["clinical"], p=2, dim=1)

        fused = torch.cat([wsi_pooled, mri, clinical], dim=1)
        fused = self.input_norm(fused)
        return self.head(self.trunk(fused))

## Intermediate fusion

Each modality gets its own `Linear` projection into a shared embedding space -- this
per-modality step is exactly what distinguishes this from early fusion. The three resulting
tokens then get *one* self-attention pass (not a deep transformer stack) so they can attend to
each other, followed by a residual connection, LayerNorm, and mean pooling. This mirrors
HIMF-Surv's own approach (per-modality projection + cross-modality attention), minus the
layer-wise aggregation this course skips.

**Why `shared_dim=256` and not HIMF-Surv's 1536:** at 1536, the attention module's
in/out-projections alone would put this model at 10x+ the parameter count of early/late fusion
-- on a ~95-patient cohort, that's an unfair comparison and an overfitting risk, not a better
model. 256 keeps all three strategies in a comparable parameter regime.

In [ ]:
class IntermediateFusionModel(nn.Module):
    def __init__(self, wsi_dim=1536, mri_dim=2048, clinical_dim=22, shared_dim=256,
                 task="classification", num_time_bins=15):
        super().__init__()
        self.abmil = ABMIL(wsi_dim, hidden_dim=128)
        self.proj_wsi = nn.Linear(wsi_dim, shared_dim)
        self.proj_mri = nn.Linear(mri_dim, shared_dim)
        self.proj_clinical = nn.Linear(clinical_dim, shared_dim)

        self.self_attn = nn.MultiheadAttention(embed_dim=shared_dim, num_heads=4, batch_first=True)
        self.attn_norm = nn.LayerNorm(shared_dim)
        self.head = build_task_head(shared_dim, task, num_time_bins)

    def forward(self, batch: dict) -> torch.Tensor:
        wsi_pooled = torch.stack([self.abmil(w) for w in batch["wsi"]])  # (B, wsi_dim)
        tokens = torch.stack([
            self.proj_wsi(wsi_pooled),
            self.proj_mri(batch["mri"]),
            self.proj_clinical(batch["clinical"]),
        ], dim=1)  # (B, 3, shared_dim)

        attn_out, _ = self.self_attn(tokens, tokens, tokens)  # one round of cross-modal attention
        tokens = self.attn_norm(tokens + attn_out)            # residual + norm
        fused = tokens.mean(dim=1)                             # (B, shared_dim)
        return self.head(fused)

## Late fusion

Each modality gets a fully independent prediction -- they never share features or an
intermediate representation. WSI still needs ABMIL to turn its patch bag into a single vector
(an unavoidable data-shape step, not a fusion choice), but there's no extra branch MLP after
it: pooled WSI, raw MRI, and raw clinical vectors go straight into their own `PredictionHead`
(itself already a 3-layer MLP, so there's no missing capacity). Only the three independent
predictions are combined at the very end, via a learned per-modality weight (three learnable
logits, softmax-normalized).

In [ ]:
class LateFusionModel(nn.Module):
    def __init__(self, wsi_dim=1536, mri_dim=2048, clinical_dim=22,
                 task="classification", num_time_bins=15):
        super().__init__()
        self.abmil = ABMIL(wsi_dim, hidden_dim=128)

        self.wsi_head = build_task_head(wsi_dim, task, num_time_bins)
        self.mri_head = build_task_head(mri_dim, task, num_time_bins)
        self.clinical_head = build_task_head(clinical_dim, task, num_time_bins)

        self.modality_logits = nn.Parameter(torch.zeros(3))  # learned combination weights

    def forward(self, batch: dict) -> torch.Tensor:
        wsi_pooled = torch.stack([self.abmil(w) for w in batch["wsi"]])

        out_wsi = self.wsi_head(wsi_pooled)
        out_mri = self.mri_head(batch["mri"])
        out_clinical = self.clinical_head(batch["clinical"])

        weights = F.softmax(self.modality_logits, dim=0)
        return weights[0] * out_wsi + weights[1] * out_mri + weights[2] * out_clinical

## Sanity-checking all six combinations

A dummy batch matching the real data's shapes (variable-length WSI bags, fixed-size MRI and
clinical vectors), run through all 3 fusion strategies x 2 tasks: check the output shape is
right, and confirm a gradient actually flows end-to-end (forward -> loss -> backward).

In [ ]:
B = 4
dummy_batch = {
    "wsi": [torch.randn(50 + 10 * i, 1536) for i in range(B)],  # variable bag size, like real WSIs
    "mri": torch.randn(B, 2048),
    "clinical": torch.randn(B, 22),
}
dummy_y_time = torch.rand(B) * 60
dummy_y_event = torch.tensor([1.0, 0.0, 1.0, 0.0])

fusion_classes = {
    "early": EarlyFusionModel,
    "intermediate": IntermediateFusionModel,
    "late": LateFusionModel,
}

results = []
for name, cls in fusion_classes.items():
    for task in ["classification", "survival"]:
        model = cls(task=task, num_time_bins=15)
        out = model(dummy_batch)
        n_params = sum(p.numel() for p in model.parameters())

        expected_shape = (B, 1) if task == "classification" else (B, 15)
        assert out.shape == expected_shape, (name, task, out.shape, expected_shape)

        if task == "classification":
            loss = F.binary_cross_entropy_with_logits(out.squeeze(1), dummy_y_event)
        else:
            y_time_bins = discretize_time(dummy_y_time, num_bins=15, max_time=60.0, device="cpu")
            loss = NLLLoss()(out, y_time_bins, dummy_y_event)
        loss.backward()

        has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
        assert has_grad, (name, task, "no gradient flowed")

        results.append({
            "fusion": name, "task": task,
            "output_shape": tuple(out.shape), "num_params": n_params,
            "dummy_loss": round(loss.item(), 4),
        })

comparison_df = pd.DataFrame(results)
comparison_df

In [ ]:
# Parameter counts should be identical between the two tasks per fusion strategy (the only
# difference between tasks is the head's output_dim, a tiny fraction of total parameters).
comparison_df.pivot(index="fusion", columns="task", values="num_params")

## Summary

All three fusion strategies x both tasks produce correctly-shaped output and train (gradients
flow end-to-end) on dummy data. Rough takeaways from the parameter counts above:

- **Early** (~1.17M params): simplest cross-modal interaction (none, until the shared trunk),
  cheapest to reason about.
- **Intermediate** (~1.40M params): richest architecture -- per-modality projection *and*
  cross-modal attention -- at a modest parameter cost over early fusion, thanks to the reduced
  `shared_dim` and single attention pass.
- **Late** (~0.43M params): fewest parameters despite three full independent heads, because
  there's no shared trunk or attention projection at all -- and the most robust to a missing
  modality, since dropping one term from the weighted sum doesn't require retraining the rest.

**Code organization:** everything above also lives in `src/fusion_models.py` (identical
definitions), so notebook 04 can import `EarlyFusionModel`, `IntermediateFusionModel`,
`LateFusionModel`, `build_task_head`, `discretize_time`, `NLLLoss`, and `concordance_index`
directly instead of redefining them.

**Next up -- `04_train_and_evaluate.ipynb`:** load the real features/labels from notebooks 01-02
-- grouping each patient's per-slide WSI feature files into slide-level training rows (patient
label copied onto every slide, MRI/clinical repeated) -- build a `Dataset`/`DataLoader`, run
stratified cross-validation for all six (fusion x task) combinations, and at evaluation time
aggregate each patient's slide-level predictions (mean probability + majority vote for
classification, mean risk score for survival) before computing AUROC / C-index.